# Gender Classification Model

**Author:** Zohaib Sheikh  
**Capstone:** TravelWise — Integrating MLOps in Travel Analytics  
**GitHub Repository:** https://github.com/zohaibsheikh007/Travel-MLOps-Capstone

## Project Summary

This notebook builds a binary classifier that predicts a TravelWise user's gender (male / female) from their public profile fields: `name`, `company`, `age`, and `code`. The trained pipeline is shipped as a Streamlit app (`gender_app.py`) and lives in the same monorepo as the flight-price regression and the hotel recommender. The intent is to learn segmentation features that can later feed personalised recommendations and pricing experiments.

We deliberately keep the model interpretable — it is a logistic regression on top of TF-IDF name features and standardised numeric attributes.

## 1. Setup

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn -q

In [ ]:
import warnings
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 2. Load and inspect

In [ ]:
df = pd.read_csv('users.csv')
print(f'Rows: {len(df)} | Columns: {df.shape[1]}')
df.head()

In [ ]:
print('Class balance:')
print(df['gender'].value_counts(normalize=True))

## 3. EDA

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df['gender'].value_counts().plot(kind='bar', ax=ax[0], color=['#4f6d7a', '#c0d6df'])
ax[0].set_title('Gender distribution'); ax[0].set_xticklabels(['Female', 'Male'], rotation=0)
sns.boxplot(x='gender', y='age', data=df, ax=ax[1])
ax[1].set_title('Age by gender')
plt.tight_layout(); plt.show()

In [ ]:
company_gender = df.groupby('company')['gender'].value_counts(normalize=True).unstack().fillna(0)
company_gender.plot(kind='bar', stacked=True, color=['#4f6d7a', '#c0d6df'])
plt.title('Gender mix per company'); plt.ylabel('share'); plt.tight_layout(); plt.show()

## 4. Feature Engineering

* Encode the target as `male=1, female=0`.
* Label-encode `company`.
* TF-IDF vectorise the user's `name` (gender signal often hides in name endings, e.g. "-a", "-y").
* Standard-scale the numeric features.

In [ ]:
df['gender_label'] = (df['gender'].str.lower() == 'male').astype(int)
X = df[['name', 'company', 'age', 'code']].copy()
y = df['gender_label']
le = LabelEncoder()
X['company_encoded'] = le.fit_transform(X['company'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocess = ColumnTransformer([
    ('name_tfidf', TfidfVectorizer(max_features=200, ngram_range=(1, 3), analyzer='char_wb'), 'name'),
    ('numeric', StandardScaler(), ['age', 'code', 'company_encoded']),
])

## 5. Model Comparison

In [ ]:
candidates = {
    'LogReg':       LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
}
rows = []
for name, clf in candidates.items():
    pipe = Pipeline([('features', preprocess), ('clf', clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    rows.append({'model': name, 'accuracy': accuracy_score(y_test, pred), 'auc': roc_auc_score(y_test, proba)})
leaderboard = pd.DataFrame(rows)
leaderboard

## 6. Hyperparameter tuning on Logistic Regression

In [ ]:
pipe = Pipeline([('features', preprocess), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
grid = {'clf__C': [0.1, 1.0, 5.0], 'clf__penalty': ['l2']}
search = GridSearchCV(pipe, grid, cv=3, scoring='roc_auc', n_jobs=-1)
search.fit(X_train, y_train)
print('Best params:', search.best_params_)
print('Best CV AUC :', round(search.best_score_, 4))
best_pipe = search.best_estimator_

## 7. Final Evaluation

In [ ]:
pred = best_pipe.predict(X_test)
proba = best_pipe.predict_proba(X_test)[:, 1]
print(f'Accuracy: {accuracy_score(y_test, pred):.4f}')
print(f'AUC     : {roc_auc_score(y_test, proba):.4f}\n')
print(classification_report(y_test, pred, target_names=['Female', 'Male']))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0],
            xticklabels=['Female', 'Male'], yticklabels=['Female', 'Male'])
ax[0].set_title('Confusion matrix'); ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Actual')
fpr, tpr, _ = roc_curve(y_test, proba)
ax[1].plot(fpr, tpr, color='steelblue'); ax[1].plot([0, 1], [0, 1], 'r--')
ax[1].set_title('ROC curve'); ax[1].set_xlabel('FPR'); ax[1].set_ylabel('TPR')
plt.tight_layout(); plt.show()

**Reasoning.** Around 67% of users in the dataset are female, so a 'predict majority class' baseline already scores ~0.67 accuracy. The character-level TF-IDF on names lifts the model meaningfully because Brazilian names like "Roberta" or "Felipe" carry strong gender signal in their suffixes. The model is honest about its limits — it would not generalise to gender-neutral names without additional features (e.g. transaction patterns), and that is documented as future work in the report.

## 8. Persist the artifacts

In [ ]:
import pickle
with open('gender_model.pkl', 'wb') as f: pickle.dump(best_pipe, f)
with open('company_encoder.pkl', 'wb') as f: pickle.dump(le, f)
print('Artifacts saved. Streamlit app: streamlit run gender_app.py')